In [ ]:
import sys
sys.path.append("..")

In [ ]:
import torch
import torch.nn.functional as F
import pandas as pd
from torch.utils.data import DataLoader
from pathlib import Path

from src.model import CardinalityEstimator
from src.dataset import GalacticBinariesDataset
from config import val_dataset_path, BATCH_SIZE

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

ckpt_path = Path("../runs/run_20260505_174120/checkpoints/best_checkpoint.pth")

model = CardinalityEstimator(max_K=10).to(device)
ckpt = torch.load(ckpt_path, map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

In [ ]:
val_ds = GalacticBinariesDataset(
    val_dataset_path,
    max_K=10,
    max_samples=10_000,
    noise=False,
    deterministic=True,
    seed=0,
)

val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
rows = []

with torch.no_grad():
    offset = 0
    for X, y in val_loader:
        X = torch.as_tensor(X, dtype=torch.float32, device=device)
        y = torch.as_tensor(y, dtype=torch.long, device=device)

        logits = model(X)
        probs = F.softmax(logits, dim=1)
        pred = probs.argmax(dim=1)
        conf = probs.max(dim=1).values
        nll = F.cross_entropy(logits, y, reduction="none")

        top2 = probs.topk(2, dim=1).values
        margin = top2[:, 0] - top2[:, 1]

        for i in range(len(y)):
            rows.append({
                "idx": offset + i,
                "true_class": y[i].item(),
                "pred_class": pred[i].item(),
                "true_K": y[i].item() + 1,
                "pred_K": pred[i].item() + 1,
                "correct": pred[i].item() == y[i].item(),
                "error": pred[i].item() - y[i].item(),
                "abs_error": abs(pred[i].item() - y[i].item()),
                "confidence": conf[i].item(),
                "margin": margin[i].item(),
                "nll": nll[i].item(),
            })

        offset += len(y)

df = pd.DataFrame(rows)
df.head()

In [ ]:
print(f"Overall accuracy: {df.correct.mean():.4f}")

In [ ]:
summary_by_k = (
    df.groupby("true_class")
    .agg(
        n=("idx", "count"),
        accuracy=("correct", "mean"),
        mae=("abs_error", "mean"),
        error=("error", "mean"),
        avg_confidence=("confidence", "mean"),
        avg_nll=("nll", "mean"),
    )
)

summary_by_k

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

cm = confusion_matrix(df.true_K, df.pred_K, labels=range(1, 11))
disp = ConfusionMatrixDisplay(cm, display_labels=range(1, 11))
disp.plot(cmap="Blues", values_format="d")
plt.xlabel("Predicted K")
plt.ylabel("True K")
plt.show()

In [ ]:
df["abs_error"].value_counts().sort_index().plot(kind="bar")
print(df["abs_error"].value_counts().sort_index())
plt.xlabel("|predicted K - true K|")
plt.ylabel("Number of examples")
plt.title("Error magnitude distribution")
plt.show()


In [ ]:
df.groupby("correct")[["confidence", "margin", "nll"]].describe()
plt.hist(df[df.correct].confidence, bins=30, alpha=0.6, label="correct")
plt.hist(df[~df.correct].confidence, bins=30, alpha=0.6, label="wrong")
plt.xlabel("Model confidence")
plt.ylabel("Count")
plt.legend()
plt.show()


In [ ]:
def plot_example(dataset, row):
    X, y = dataset[int(row.idx)]

    fig, axes = plt.subplots(X.shape[0], 1, figsize=(12, 6), sharex=True)
    for c, ax in enumerate(axes):
        ax.plot(X[c])
        ax.set_ylabel(f"ch {c}")

    fig.suptitle(
        f"idx={row.idx} | true K={row.true_K} | pred K={row.pred_K} | "
        f"conf={row.confidence:.3f} | abs err={row.abs_error}"
    )
    plt.show()

hard_wrong = df[~df.correct].sort_values("confidence", ascending=False).head(5)
easy_correct = df[df.correct].sort_values("confidence", ascending=False).head(5)

plot_example(val_ds, hard_wrong.iloc[0])
plot_example(val_ds, easy_correct.iloc[0])


In [ ]:
def evaluate_with_noise_sigma(model, dataset, sigma):
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)
    rows = []

    with torch.no_grad():
        for X, y in loader:
            X = torch.as_tensor(X, dtype=torch.float32, device=device)
            y = torch.as_tensor(y, dtype=torch.long, device=device)

            if sigma > 0:
                X = X + sigma * torch.randn_like(X)

            pred = model(X).argmax(dim=1)
            abs_err = (pred - y).abs()

            rows.append(pd.DataFrame({
                "true_K": y.cpu().numpy() + 1,
                "pred_K": pred.cpu().numpy() + 1,
                "correct": (pred == y).cpu().numpy(),
                "abs_error": abs_err.cpu().numpy(),
                "sigma": sigma,
            }))

    return pd.concat(rows, ignore_index=True)

noise_results = pd.concat([
    evaluate_with_noise_sigma(model, val_ds, sigma)
    for sigma in [0.0, 0.25, 0.5, 1.0, 2.0]
])

noise_results.groupby("sigma").agg(
    accuracy=("correct", "mean"),
    mae=("abs_error", "mean"),
)


In [ ]:
# See available physical parameter names
val_ds.attr_names


In [ ]:
param_rows = []

for row in df.itertuples():
    sampled_indices, target = val_ds.fixed_mixtures[row.idx]

    item = {
        "idx": row.idx,
        "true_K": row.true_K,
        "pred_K": row.pred_K,
        "correct": row.correct,
        "abs_error": row.abs_error,
    }

    for name in val_ds.attr_names:
        values = getattr(val_ds, name)[sampled_indices]
        item[f"{name}_mean"] = values.mean()
        item[f"{name}_std"] = values.std()
        item[f"{name}_min"] = values.min()
        item[f"{name}_max"] = values.max()

    param_rows.append(item)

param_df = pd.DataFrame(param_rows)
param_df.head()


In [ ]:
param_df.groupby("correct").mean(numeric_only=True).T

In [ ]:
pd.DataFrame({
    "true_count": df.true_K.value_counts().sort_index(),
    "pred_count": df.pred_K.value_counts().sort_index(),
}).plot(kind="bar")
plt.xlabel("K")
plt.ylabel("Number of examples")
plt.title("True vs predicted K distribution")
plt.show()
